# 집계_3_영업이익잠식
같은 운임 인상이 **대기업과 중소기업의 영업이익**을 얼마나 깎는지 비교합니다.

- **운임부담률** = 추가 운임 부담(집계_1) ÷ 같은 기간 그 지역 수출액(관세청) → 수출 100원당 운임을 몇 원 더 냈나
- **영업이익잠식률** = 운임부담률 ÷ 위기 직전 4개 분기 매출액영업이익률(한국은행, 규모별)

- 입력: `financedata/집계_1_총피해액.xlsx`, `finance/142_DT_B10066_….csv`(대륙 구성 국가), `finance/D10_관세청_국가별수출_월별.csv`(없으면 API로 받음), 한국은행 ECOS API
- 필요: `finance/ecos_key.txt`, (D10이 없을 때) `finance/data_go_kr_key.txt`
- 출력: `집계_3_영업이익잠식.xlsx` (설명 / 영업이익잠식 / 영업이익률_분기)

In [ ]:
from pathlib import Path
import pandas as pd

DESK = Path.home() / "OneDrive" / "Desktop"
DATA = DESK / "financedata"   # 입력 데이터
KEYS = DESK / "finance"       # API 키, KOSIS·관세청 원자료
SAVE = DESK / "finance"       # 결과 저장 위치 

import time
import xml.etree.ElementTree as ET
import requests

ECOS_KEY = (KEYS / "ecos_key.txt").read_text().strip()
INDUSTRY = {"2000": "제조업", "1000": "전산업"}
SIZE = {"L": "대기업", "M": "중소기업"}
PERIODS = {  # (위기, 기간): (해수부 지역, 시작월, 끝월, 이익률 기준 분기)
    ("홍해", "첫 분기"): ("유럽 지역", "2023-12", "2024-02", ["2022Q4", "2023Q1", "2023Q2", "2023Q3"]),
    ("홍해", "첫 1년"): ("유럽 지역", "2023-12", "2024-11", ["2022Q4", "2023Q1", "2023Q2", "2023Q3"]),
    ("호르무즈", "첫 분기"): ("중동 지역", "2026-03", "2026-05", ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]),
    ("호르무즈", "6개월"): ("중동 지역", "2026-03", "2026-08", ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]),
}

## 1. 한국은행 매출액영업이익률 (ECOS 502Y002, 분기, 업종 × 규모)

In [ ]:
rows = []
for ic, iname in INDUSTRY.items():
    for sc, sname in SIZE.items():
        url = (f"https://ecos.bok.or.kr/api/StatisticSearch/{ECOS_KEY}/json/kr/1/500/502Y002/Q/2022Q1/2026Q2/"
               f"{ic}/{sc}/3000")    # 3000 = 매출액영업이익률
        for x in requests.get(url, timeout=60).json().get("StatisticSearch", {}).get("row", []):
            rows.append({"업종": iname, "규모": sname, "분기": x["TIME"], "매출액영업이익률_%": float(x["DATA_VALUE"])})
opm = pd.DataFrame(rows)
opm.pivot_table(index="분기", columns=["업종", "규모"], values="매출액영업이익률_%")

## 2. 지역 수출액 (관세청 월별) — 유럽(러시아·벨라루스 제외)·중동 구성 국가는 KOSIS 대륙 분류를 따름

In [ ]:
kos = pd.read_csv(KEYS / "142_DT_B10066_20260930070146.csv", encoding="cp949")
names = list(dict.fromkeys(kos.iloc[:, 0].str.strip()))   # 한 나라가 여러 줄이라 순서 유지하며 중복 제거
CONTS = ["계", "아시아", "중동", "유럽", "북미", "중남미", "아프리카", "남아프리카", "중앙아프리카", "대양주", "기타지역", "기타국"]


def members(cont):
    out = []
    for n in names[names.index(cont) + 1:]:
        if n in CONTS:
            break
        out.append(n)
    return out


RENAME = {"터키": "튀르키예", "세르비아공화국": "세르비아", "마세도니아(마케도니아)": "마세도니아"}  # KOSIS → 관세청 이름
europe = [RENAME.get(n, n) for n in members("유럽") if n not in ("러시아 연방", "벨라루스")]
mideast = members("중동")

d10 = KEYS / "D10_관세청_국가별수출_월별.csv"
if not d10.exists():   
    key = (KEYS / "data_go_kr_key.txt").read_text().strip()
    recs = []
    for p in pd.period_range("2022-01", "2026-08", freq="M"):
        r = requests.get("https://apis.data.go.kr/1220000/nationtrade/getNationtradeList",
                         params={"serviceKey": key, "strtYymm": p.strftime("%Y%m"), "endYymm": p.strftime("%Y%m")}, timeout=90)
        recs += [{c.tag: c.text for c in it} for it in ET.fromstring(r.content).findall(".//item")]
        time.sleep(0.3)
    tmp = pd.DataFrame(recs).rename(columns={"year": "연월", "statCd": "국가코드", "statCdCntnKor1": "국가", "expDlr": "수출액_달러"})
    tmp["수출액_달러"] = pd.to_numeric(tmp["수출액_달러"])
    tmp.to_csv(d10, index=False, encoding="utf-8-sig")

try:
    cu = pd.read_csv(d10, dtype={"연월": str})
except UnicodeDecodeError:
    cu = pd.read_csv(d10, dtype={"연월": str}, encoding="cp949")
cu["월"] = cu["연월"].map(lambda v: f"{float(v):.2f}").str.replace(".", "-")   # 엑셀 저장 후 '2022.1'(10월)로 바뀐 경우 처리
reg_exp = {"유럽 지역": cu[cu["국가"].isin(europe)].groupby("월")["수출액_달러"].sum(),
           "중동 지역": cu[cu["국가"].isin(mideast)].groupby("월")["수출액_달러"].sum()}
print("유럽", len(europe), "개국 / 중동", len(mideast), "개국")

## 3. 운임부담률과 영업이익잠식률

In [ ]:
burden = pd.read_excel(DATA / "집계_1_총피해액.xlsx", sheet_name="월별")

res = []
for (crisis, pname), (area, s, e, base_q) in PERIODS.items():
    b = burden[(burden["위기"] == crisis) & (burden["월"] >= s) & (burden["월"] <= e)]
    exp_krw = (reg_exp[area].reindex(b["월"]).values * b["환율_원per달러"].values).sum()
    burden_krw = b["추가운임부담_억원"].sum() * 1e8
    rate = burden_krw / exp_krw * 100
    for iname in INDUSTRY.values():
        for sname in SIZE.values():
            o = opm[(opm["업종"] == iname) & (opm["규모"] == sname) & (opm["분기"].isin(base_q))]["매출액영업이익률_%"].mean()
            res.append({"위기": crisis, "기간": pname, "기간_월": f"{s}~{e}", "지역": area,
                        "추가운임부담_억원": round(burden_krw / 1e8), "지역수출액_억원": round(exp_krw / 1e8),
                        "운임부담률_%": round(rate, 2), "업종": iname, "규모": sname,
                        "이익률기준분기": f"{base_q[0]}~{base_q[-1]}", "영업이익률_%": round(o, 2),
                        "영업이익잠식률_%": round(rate / o * 100, 1)})
res = pd.DataFrame(res)
res[res["업종"] == "제조업"]

## 4. 저장

In [ ]:
notes = pd.DataFrame([
    ("목적", "같은 운임 인상이 대기업과 중소기업 이익에 주는 타격 비교"),
    ("운임부담률", "KCCI 기준 추가 운임 부담(집계_1) ÷ 같은 달들의 그 지역 수출액(관세청, 월 환율로 원화). 대·중소 동일 가정"),
    ("지역 수출액", "유럽 = KOSIS 유럽 대륙 구성국(러시아·벨라루스 제외), 중동 = KOSIS 중동 구성국. 관세청 국가별 월별 수출"),
    ("영업이익률", "한국은행 ECOS 502Y002 기업경영분석(분기) 매출액영업이익률, 위기 직전 4개 분기 평균. "
                 "분기 기업경영분석은 외부감사 대상 기업 표본 기준(작은 중소기업은 포함 안 됨)"),
    ("영업이익잠식률", "운임부담률 ÷ 영업이익률 × 100 = 늘어난 운임을 가격에 전혀 넘기지 못할 때 영업이익의 몇 %가 사라지나"),
    ("해석", "중소기업이 대기업보다 운임을 더 많이 낸다는 뜻이 아님. 같은 비율로 내도 이익이 얇아 더 크게 잠식된다는 뜻"),
    ("가격 전가", "무역협회(2026.7): 물류비 인상분을 가격에 전혀 반영 못함 32.9%, 20% 미만 반영 78.1% → 잠식률은 이 기업들에 해당"),
], columns=["항목", "내용"])
out = SAVE / "집계_3_영업이익잠식.xlsx"
with pd.ExcelWriter(out) as w:
    notes.to_excel(w, sheet_name="설명", index=False)
    res.to_excel(w, sheet_name="영업이익잠식", index=False)
    opm.pivot_table(index="분기", columns=["업종", "규모"], values="매출액영업이익률_%").to_excel(w, sheet_name="영업이익률_분기")
print("저장:", out)